
<a id='chap-coursdeep8'></a>

 Work in progress

Contenu rédigé à partir de la dernière section des diapositives
coursSSL.pdf, faute de source préexistante pour cette séance.
Susceptible d’être ajusté.

Les informations pratiques concernant le déroulement de l’unité d’enseignement RCP209 « Apprentissage, réseaux de neurones et modèles graphiques » au Cnam se trouvent dans [ce préambule](preambule.html).

# XV.1 - Cours

[[Diapositives du cours](http://cedric.cnam.fr/vertigo/Cours/ml2/docs/coursSSL.pdf)]

Dans le [chapitre précédent](../session14/course.ipynb#chap-coursdeep7), nous avons vu comment l’apprentissage auto-supervisé permet d’apprendre des représentations utiles à partir de données non étiquetées, en se passant d’une supervision humaine explicite : tâches prétextes, apprentissage contrastif (SimCLR, MoCo), auto-distillation (DINO) et modélisation masquée (MAE). Ces méthodes partagent un même constat : on n’entraîne plus un modèle pour une unique tâche, mais on apprend d’abord une représentation générale (le **pré-entraînement**), que l’on adapte ensuite à de nombreuses tâches *downstream* différentes. Les **modèles de fondation** (*foundation models*) poussent cette logique à son terme.

## Qu’est-ce qu’un modèle de fondation ?

Un **modèle de fondation** est un modèle entraîné sur des données à très large échelle (généralement à l’aide de l’apprentissage auto-supervisé), et qui peut ensuite être **adapté** à un très grand nombre de tâches, de domaines, voire de modalités différentes (Bommasani et al., 2021). Les ingrédients-clés en sont :

- des **données d’entraînement à grande échelle** et peu ou pas curées (du texte, des images, voire plusieurs modalités simultanément, collectées à l’échelle du Web) ;  
- un **pré-entraînement auto-supervisé**, qui ne nécessite donc pas d’annotation humaine à cette échelle ;  
- un mécanisme d”**adaptation** à une tâche particulière, qui peut être un simple *prompt*, un *fine-tuning* léger, ou un ré-entraînement plus complet ;  
- idéalement, la capacité d’intégrer **plusieurs modalités** (par exemple, images et texte) dans une représentation commune.  


Le changement de paradigme est résumé par le constat suivant : l’objectif n’est plus de résoudre une tâche donnée, mais d’apprendre une fondation réutilisable, sur laquelle de nombreuses tâches pourront ensuite s’appuyer.

## Modèles vision-langage : CLIP

### Principe

**CLIP** (*Contrastive Language-Image Pre-training*, Radford et al., 2021) est l’un des modèles de fondation les plus influents pour la vision par ordinateur. Son objectif est d’apprendre conjointement des représentations d’images et de texte, dans un **espace d’embedding commun**, tel qu’une image et sa légende soient proches dans cet espace, et qu’une image et une légende sans rapport en soient éloignées.

L’architecture est constituée de deux encodeurs indépendants :

- un **encodeur d’images** $ f_{\text{img}} $ (par exemple, un ResNet ou un Vision Transformer), qui produit un embedding $ z_{\text{img}} = f_{\text{img}}(x_{\text{img}}) $ ;  
- un **encodeur de texte** $ f_{\text{text}} $ (un Transformer), qui produit un embedding $ z_{\text{text}} = f_{\text{text}}(x_{\text{text}}) $.  


L’entraînement est directement une instanciation de l’apprentissage contrastif vu au [chapitre précédent](../session14/course.ipynb#chap-coursdeep7), mais **entre deux modalités** plutôt qu’entre deux vues augmentées d’une même image :

1. On échantillonne un batch de $ N $ paires (image, légende) collectées sur le Web (CLIP a été entraîné sur 400 millions de telles paires).  
1. On encode toutes les images et tous les textes du batch, et l’on calcule la similarité cosinus entre chacune des $ N \times N $ combinaisons (image, texte) possibles.  
1. Pour chaque image, la légende qui lui correspond réellement doit obtenir la similarité la plus élevée parmi les $ N $ légendes du batch (et réciproquement pour chaque texte) : on optimise une perte InfoNCE **symétrique**, à la fois image→texte et texte→image.  

### Question

En quoi la construction des paires positives et négatives de CLIP diffère-t-elle de celle de SimCLR (vu au chapitre précédent) ? Qu’ont-elles en commun ?

### Classification *zero-shot*

Une fois entraîné, CLIP peut être utilisé pour classifier des images dans **n’importe quel ensemble de catégories**, y compris des catégories jamais vues pendant l’entraînement, sans aucun réglage fin (*zero-shot classification*) :

1. Pour chaque nom de classe possible (par exemple, « chat », « chien », …), on construit une phrase-modèle (*prompt*), par exemple *« a photo of a {classe} »*.  
1. On encode chacune de ces phrases avec l’encodeur de texte de CLIP, pour obtenir un embedding par classe.  
1. On encode l’image à classifier avec l’encodeur d’image.  
1. On calcule la similarité entre l’embedding de l’image et l’embedding de chaque classe : la classe prédite est celle dont l’embedding textuel est le plus similaire à l’image.  


Cette capacité rend possible, par le même mécanisme, la **reconnaissance à vocabulaire ouvert** (*open-vocabulary recognition*) : détecter ou segmenter des catégories qui n’ont jamais été vues lors de l’entraînement, simplement en les décrivant par du texte, ainsi que la recherche d’images par texte et de texte par image.

### Question

Sur certains jeux de données (par exemple, MNIST ou des jeux de données très spécialisés comme la classification de cellules), CLIP en *zero-shot* obtient des performances nettement inférieures à un classifieur entraîné spécifiquement (supervisé) sur ces données. Comment l’expliquez-vous ?

## Au-delà de CLIP : les modèles vision-langage

CLIP n’est qu’un point de départ : de nombreux **modèles vision-langage** (*Vision-Language Models*, VLM) ont depuis étendu ce principe, avec des stratégies d’entraînement variées, parfois combinées :

- **contrastive**, comme CLIP lui-même ;  
- **par modélisation masquée**, en s’inspirant de MAE mais appliquée conjointement à l’image et au texte ;  
- **générative**, où le modèle apprend à produire du texte conditionné sur une image (légendage, réponse à des questions visuelles), en s’appuyant sur un grand modèle de langage (LLM) pré-entraîné séparément, connecté à un encodeur visuel.  


Des architectures comme **BLIP** ou **FLAVA** combinent plusieurs de ces objectifs au sein d’un même modèle, tandis que des architectures récentes comme **Qwen-VL** connectent un Vision Transformer pré-entraîné à un grand modèle de langage, avec un entraînement en plusieurs étapes (pré-entraînement contrastif, pré-entraînement multi-tâche, puis réglage fin supervisé). Cette diversité illustre un phénomène plus général : vision, langage et modèles multimodaux convergent aujourd’hui vers un paradigme d’entraînement commun, combinant pré-entraînement à grande échelle et adaptation à la tâche.

## Conclusion

Ce chapitre conclut la partie de ce cours consacrée à l’apprentissage profond. Le fil directeur des dernières séances peut se résumer ainsi :

- l’apprentissage auto-supervisé permet d’apprendre des représentations sans annotation humaine, en résolvant une tâche prétexte ou un objectif contrastif, de distillation, ou de reconstruction masquée ;  
- le pré-entraînement à très grande échelle sur ce principe donne naissance à des **modèles de fondation**, capables d’être adaptés à une grande diversité de tâches, y compris en combinant plusieurs modalités ;  
- ce changement de paradigme — ne plus entraîner un modèle par tâche, mais apprendre une fondation réutilisable — est aujourd’hui au cœur de la plupart des systèmes d’apprentissage profond à l’état de l’art, en vision comme en traitement du langage.  